In [2]:
import pandas as pd

In [3]:
df = pd.read_json('combined_SOsurvey_data.json')

In [4]:
df_desire = df[['year_of_survey', 'LanguageDesireNextYear']].copy()

df_desire = df_desire[df_desire['LanguageDesireNextYear'].notna()]
df_desire = df_desire[df_desire['LanguageDesireNextYear'] != 'None']

# แยกภาษาและกระจาย
df_desire['Language'] = df_desire['LanguageDesireNextYear'].str.split(';')
df_desire = df_desire.explode('Language')
df_desire['Language'] = df_desire['Language'].str.strip()

# นับจำนวน(แบ่งตามปีและภาษา)
desire_count = df_desire.groupby(['year_of_survey', 'Language']).size().reset_index(name='count')

In [ ]:
# สร้างคอลัมน์ใหม่เป็นตัวพิมพ์ใหญ่ทั้งหมด ('COBOL' กับ 'Cobol')
desire_count['Language_upper'] = desire_count['Language'].str.upper()

# รวมตามภาษาใหม่
total_norm = desire_count.groupby('Language_upper', as_index=False)['count'].sum()
total_norm.rename(columns={'Language_upper': 'Language', 'count': 'total_count'}, inplace=True)


print(total_norm.head(10))

# Export to JSON
json_total_norm = total_norm.to_json(orient='records', force_ascii=False, indent=2)
#print(json_total_norm)

                  Language  total_count
0                      ADA         3458
1                     APEX         1740
2                      APL         3166
3                 ASSEMBLY        36168
4               BASH/SHELL        54570
5  BASH/SHELL (ALL SHELLS)        81370
6    BASH/SHELL/POWERSHELL        23456
7                        C        93002
8                       C#       169970
9                      C++       134496


In [11]:
with open('SO_survey_total_prolang_desire.json', 'w', encoding='utf-8') as f:
    f.write(json_total_norm)

In [12]:
# นับตาม year + Language (normalize เป็นตัวพิมพ์ใหญ่)
desire_count['Language_upper'] = desire_count['Language'].str.upper()

year_norm = desire_count.groupby(['year_of_survey', 'Language_upper'], as_index=False)['count'].sum()
year_norm.rename(columns={'Language_upper': 'Language'}, inplace=True)

# Loop แต่ละปีแล้ว export แยกไฟล์
for year, group in year_norm.groupby('year_of_survey'):
    df_year = group[['year_of_survey', 'Language', 'count']].reset_index(drop=True)
    json_out = df_year.to_json(orient='records', force_ascii=False, indent=2)
    filename = f'SO_survey_{year}_desire.json'
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(json_out)
    print(f'Saved: {filename} ({len(df_year)} rows)')


Saved: SO_survey_2020_desire.json (25 rows)
Saved: SO_survey_2021_desire.json (38 rows)
Saved: SO_survey_2022_desire.json (42 rows)
Saved: SO_survey_2023_desire.json (51 rows)
Saved: SO_survey_2024_desire.json (49 rows)
Saved: SO_survey_2025_desire.json (42 rows)
